You are a Senior Data Engineer at Stripe. You are building a pipeline that joins a massive transactions table (5
billion rows) with a merchants lookup table (10 million rows). During production runs, you observe that several
executors are processing 50× more data than others, causing the job to stall for hours. Profiling reveals that
approximately 40% of all transactions belong to 5 "whale" merchants (e.g., Amazon, Walmart), creating extreme
key skew.
Implement a salted join to redistribute the skewed keys and compare it against a naive join. The solution must: 1.
Identify the top-N skewed keys dynamically (not hardcoded). 2. Apply salting only to the skewed keys; use a regular
join for non-skewed keys. 3. Produce the same result as the un-skewed join.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

txn_data = [
    ("t001", "M_AMAZON", 59.99),
    ("t002", "M_AMAZON", 120.00),
    ("t003", "M_SMALL_CO", 14.50),
    ("t004", "M_WALMART", 89.00),
    ("t005", "M_AMAZON", 22.75),
]

merchant_data = [
    ("M_AMAZON", "Amazon", "Retail"),
    ("M_WALMART", "Walmart", "Retail"),
    ("M_SMALL_CO", "Small Co", "Services")
]
transaction = spark.createDataFrame(txn_data, ["txn_id", "merchant_id", "amount"])
merchants = spark.createDataFrame(merchant_data, ["merchant_id", "merchant_name", "category"])

transaction.display()
merchants.display()

In [0]:
SALT_BUCKETS = 50 # no. of salt partitions for skewed keys
SKEW_THRESHOLD_PCT = 0.01 # keys accounting for >1% of total rows are "skewed"

#Dynamically identify skewed keys
total_txn_count = transaction.count()
skew_threshold = total_txn_count * SKEW_THRESHOLD_PCT

key_counts = (
    transaction.groupBy("merchant_id").agg(F.count("*").alias("cnt"))
)
key_counts.display()


In [0]:
#collect only the skewed merchant IDs - small list, safe to collect()
skewed_keys = [
    row.merchant_id for row in key_counts.filter(F.col("cnt") >= skew_threshold).select("merchant_id").collect()
]
print(f"Identified {len(skewed_keys)} skewed_keys: {skewed_keys}")

In [0]:
#split transactions into skewed and normal subsets
txn_skewed = transaction.filter(F.col("merchant_id").isin(skewed_keys))
txn_normal = transaction.filter(~F.col("merchant_id").isin(skewed_keys))

#salt the skewed transaction (random bucket 0 to n-1)
txn_skewed_salted = txn_skewed.withColumn("salt", (F.rand() * SALT_BUCKETS).cast(IntegerType())).withColumn("salted_merchant_id", F.concat_ws("_", F.col("merchant_id"), F.col("salt").cast("string")))

txn_skewed_salted.display()

In [0]:
# explode the merchants lookup for all salt values
# create an array [0,1,2..., SALT_BUCKETS-1] using sequence()

merchants_exploded = (
    merchants.filter(F.col("merchant_id").isin(skewed_keys)).withColumn("salt_array", F.sequence(F.lit(0), F.lit(SALT_BUCKETS - 1))).withColumn("salt", F.explode("salt_array")).withColumn("salted_merchant_id", F.concat_ws("_", F.col("merchant_id"), F.col("salt").cast("string"))).drop("salt_array","salt")
)

# join the skewed subset using the salted key
result_skewed = (
    txn_skewed_salted.join(merchants_exploded, on="salted_merchant_id", how="left").drop("salted_merchant_id", "salt").select("txn_id", txn_skewed_salted["merchant_id"], "amount", "merchant_name", "category")
)
result_skewed.display()



In [0]:
# normal join for non-skewed keys (broadcast if merchants fits in RAM)
merchants_normal = merchants.filter(~F.col("merchant_id").isin(skewed_keys))
result_normal = txn_normal.join(F.broadcast(merchants_normal), on="merchant_id", how="left").select("txn_id", "merchant_id", "amount", "merchant_name", "category")
final_result = result_skewed.unionByName(result_normal).orderBy("txn_id")
final_result.display()

This notebook demonstrates **salted join optimization** to handle extreme key skew in large-scale data processing.

Problem Statement
When joining a massive transactions table (5B rows) with a merchants lookup table (10M rows), a few "whale" merchants (Amazon, Walmart) account for 40% of transactions, causing severe data skew. This makes some executors process 50× more data than others, stalling the job for hours.

Solution Approach

The notebook implements a **hybrid salted join strategy**:

**1. Dynamic Skew Detection (Cells 3-4)**
* Counts transactions per merchant
* Identifies keys exceeding 1% of total volume as "skewed"
* Dynamically determines skewed keys (not hardcoded)

**2. Data Partitioning (Cell 5)**
* Splits transactions into skewed vs. non-skewed subsets
* Adds random salt (0-49) to skewed transactions, creating keys like `M_AMAZON_23`

**3. Dual Join Strategy (Cells 6-7)**
* **Skewed keys**: Explodes merchants lookup across all salt values (50 copies per merchant), then joins on salted key — this distributes the heavy merchants across 50 partitions
* **Non-skewed keys**: Uses efficient broadcast join (no salting overhead needed)
* Unions both results for final output

Key Benefit
This approach parallelizes work for hot keys while avoiding unnecessary overhead for normal keys, dramatically reducing job runtime by eliminating the executor imbalance.